# 📘 Modèle Pyomo généré automatiquement

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory


## Définition du modèle

In [ ]:
from pyomo.environ import *

model = ConcreteModel()

model.MATIERES = Set(initialize=['Epinette', 'Sapin'])
model.PRODUITS = Set(initialize=['Planche', 'Contreplaque'])
model.ARC = Set(dimen=2, initialize=[(i,j) for i in model.MATIERES for j in model.PRODUITS])
model.dispo = Param(model.MATIERES, initialize={'Epinette': 150000.0, 'Sapin': 200000.0}, within=NonNegativeReals)
model.gain = Param(model.PRODUITS, initialize={'Planche': 200.0, 'Contreplaque': 300.0}, within=NonNegativeReals)
model.engagement = Param(model.PRODUITS, initialize={'Planche': 40.0, 'Contreplaque': 30.0}, within=NonNegativeReals)
model.utilisation = Param(model.MATIERES, model.PRODUITS, initialize={('Epinette', 'Planche'): 1000.0, ('Epinette', 'Contreplaque'): 2000.0, ('Sapin', 'Planche'): 1200.0, ('Sapin', 'Contreplaque'): 4000.0}, within=NonNegativeReals)
model.X = Var(model.PRODUITS, domain=NonNegativeReals)
def rule_for_0(model, p):
    return model.X[p] >= model.engagement[p]
model.c_for_0 = Constraint(model.PRODUITS, rule=rule_for_0)
def rule_for_1(model, m):
    return sum(model.utilisation[m,p] * model.X[p] for p in model.PRODUITS) <= model.dispo[m]
model.c_for_1 = Constraint(model.MATIERES, rule=rule_for_1)
model.obj = Objective(expr=sum(model.gain[p] * model.X[p] for p in model.PRODUITS), sense=maximize)

## Résolution du modèle

In [ ]:
solver = SolverFactory('gurobi')
result = solver.solve(model, tee=True)
print('Solver status:', result.solver.status)
print('Termination condition:', result.solver.termination_condition)


## Valeurs optimales des variables

In [ ]:
for v in model.component_objects(Var, active=True):
    print(f'Variable set: {v}')
    for index in v:
        print(f'   {index} = {v[index].value}')
